In [1]:
import os

parent_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
dataset_dir = os.path.abspath(os.path.join(parent_dir, os.pardir)) + '/dataset/dict_all'

with open(f'{dataset_dir}/entity_all_1.txt') as f:
    entities = set()
    for line in f:
        line = line.strip()
        if not line:
            continue
        first = line.split()[0]
        entities.add(first)
        
with open(f'{dataset_dir}/entity_all_2.txt') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        first = line.split()[0]
        entities.add(first)

In [15]:
from transformers import AutoTokenizer, AutoModelForMaskedLM, pipeline
import torch
import torch.nn.functional as F

def correct_ocr_text(text, named_entities=entities, threshold=0.2, model_name="dbmdz/bert-base-italian-xxl-cased"):
    """
    Correct OCR errors in Italian text using BERT.
    
    Args:
        text (str): OCR text to correct
        named_entities (set): set of words to preserve (optional)
        threshold (float): probability below which a token is considered wrong
        model_name (str): Hugging Face model to use

    Returns:
        str: corrected text
    """
    if named_entities is None:
        named_entities = set()
    
    # Load BERT masked LM
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForMaskedLM.from_pretrained(model_name)
    model.eval()
    
    # Tokenize text into whitespace tokens
    tokens = text.split()
    
    # Step 1: Detect suspicious tokens
    suspicious_indices = []
    for i, token in enumerate(tokens):
        if token in named_entities:
            continue  # preserve named entities
        
        # Mask the token temporarily
        masked_tokens = tokens.copy()
        masked_tokens[i] = "[MASK]"
        masked_text = " ".join(masked_tokens)
        inputs = tokenizer(masked_text, return_tensors="pt")
        
        with torch.no_grad():
            outputs = model(**inputs)
        
        # Find mask index
        mask_index = (inputs['input_ids'][0] == tokenizer.mask_token_id).nonzero(as_tuple=True)[0]
        
        # Get softmax probability for original token
        mask_logits = outputs.logits[0, mask_index, :]
        token_id = tokenizer.convert_tokens_to_ids(token)
        if token_id is None:
            token_prob = 0.0  # token not in vocab → likely OCR error
        else:
            probs = F.softmax(mask_logits, dim=-1)
            token_prob = probs[0, token_id].item()
        
        # Mark as suspicious if probability is below threshold
        if token_prob < threshold:
            suspicious_indices.append(i)
    
    # Step 2: Mask all suspicious tokens at once
    masked_tokens = tokens.copy()
    for i in suspicious_indices:
        masked_tokens[i] = "[MASK]"
    masked_text = " ".join(masked_tokens)
    
    # Step 3: Predict replacements
    inputs = tokenizer(masked_text, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Replace masked tokens with BERT predictions
    input_ids = inputs['input_ids'][0]
    mask_positions = (input_ids == tokenizer.mask_token_id).nonzero(as_tuple=True)[0]
    for idx, mask_idx in enumerate(mask_positions):
        mask_logits = outputs.logits[0, mask_idx, :]
        predicted_id = mask_logits.argmax(dim=-1).item()
        predicted_token = tokenizer.convert_ids_to_tokens(predicted_id)
        masked_tokens[suspicious_indices[idx]] = predicted_token
    
    # Reconstruct corrected sentence
    corrected_text = " ".join(masked_tokens)
    return corrected_text

# Example usage
text = "Terracini prende infine la parola per ammonire j] governo che l 'opposizione vigilerà."
corrected = correct_ocr_text(text,)
print(corrected)

Some weights of the model checkpoint at dbmdz/bert-base-italian-xxl-cased were not used when initializing BertForMaskedLM: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight']
- This IS expected if you are initializing BertForMaskedLM from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForMaskedLM from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


Terracini prende subito la parola per spiegare il , che l ’ .


In [5]:
corrected_tokens

['Terracini',
 'prende',
 'prende',
 'la',
 'parola',
 'per',
 'per',
 'ammonire',
 'governo',
 'che',
 'l',
 'che',
 "l'opposizione.."]